# 01 기초: 원궤도와 Walker-Delta 배치

논문의 `i:t/p/f` 표기, RAAN(Right Ascension of the Ascending Node, 승교점 적경), AoL(Argument of Latitude, 위도인수)을 Python 표준 라이브러리로 생성한다.

In [1]:
from math import pi, sqrt

MU_EARTH_KM3_S2 = 398600.4418
R_EARTH_KM = 6378.137
altitude_km = 500
semi_major_axis_km = R_EARTH_KM + altitude_km
mean_motion_rad_s = sqrt(MU_EARTH_KM3_S2 / semi_major_axis_km**3)
period_s = 2*pi/mean_motion_rad_s
print(f'500 km 원궤도 주기: {period_s/60:.2f} min')

500 km 원궤도 주기: 94.62 min


In [2]:
def walker_delta(inclination_deg, total, planes, phase):
    if total % planes:
        raise ValueError('전체 위성 수는 궤도면 수로 나누어져야 합니다')
    if not 0 <= phase <= planes-1:
        raise ValueError('phase f는 0..p-1 범위여야 합니다')
    per_plane = total // planes
    slots = []
    for j in range(planes):
        raan_deg = 360*j/planes
        for k in range(per_plane):
            aol_deg = (360*k/per_plane + 360*phase*j/(per_plane*planes)) % 360
            slots.append({'plane':j, 'satellite':k, 'i_deg':inclination_deg,
                          'raan_deg':raan_deg, 'aol_deg':aol_deg})
    return slots

w5 = walker_delta(80,30,5,0)
w6 = walker_delta(80,30,6,5)
w5[:8], w6[:6]

([{'plane': 0, 'satellite': 0, 'i_deg': 80, 'raan_deg': 0.0, 'aol_deg': 0.0},
  {'plane': 0, 'satellite': 1, 'i_deg': 80, 'raan_deg': 0.0, 'aol_deg': 60.0},
  {'plane': 0, 'satellite': 2, 'i_deg': 80, 'raan_deg': 0.0, 'aol_deg': 120.0},
  {'plane': 0, 'satellite': 3, 'i_deg': 80, 'raan_deg': 0.0, 'aol_deg': 180.0},
  {'plane': 0, 'satellite': 4, 'i_deg': 80, 'raan_deg': 0.0, 'aol_deg': 240.0},
  {'plane': 0, 'satellite': 5, 'i_deg': 80, 'raan_deg': 0.0, 'aol_deg': 300.0},
  {'plane': 1, 'satellite': 0, 'i_deg': 80, 'raan_deg': 72.0, 'aol_deg': 0.0},
  {'plane': 1,
   'satellite': 1,
   'i_deg': 80,
   'raan_deg': 72.0,
   'aol_deg': 60.0}],
 [{'plane': 0, 'satellite': 0, 'i_deg': 80, 'raan_deg': 0.0, 'aol_deg': 0.0},
  {'plane': 0, 'satellite': 1, 'i_deg': 80, 'raan_deg': 0.0, 'aol_deg': 72.0},
  {'plane': 0, 'satellite': 2, 'i_deg': 80, 'raan_deg': 0.0, 'aol_deg': 144.0},
  {'plane': 0, 'satellite': 3, 'i_deg': 80, 'raan_deg': 0.0, 'aol_deg': 216.0},
  {'plane': 0, 'satellite': 4, 'i_

In [3]:
# 배치 불변조건 검증
assert len(w5)==len(w6)==30
assert sorted({x['raan_deg'] for x in w5}) == [0,72,144,216,288]
assert len([x for x in w5 if x['plane']==0]) == 6
assert len([x for x in w6 if x['plane']==0]) == 5
print('80°:30/5/0과 80°:30/6/5 배치 검증 통과')

80°:30/5/0과 80°:30/6/5 배치 검증 통과


In [4]:
# 논문 Table 1의 발사당/전체 질량 확인
designs = {
    '5 planes': {'mass_per_sat_kg':400, 'sat_per_plane':6, 'planes':5},
    '6 planes': {'mass_per_sat_kg':500, 'sat_per_plane':5, 'planes':6},
}
for name,d in designs.items():
    per_launch = d['mass_per_sat_kg']*d['sat_per_plane']
    total = per_launch*d['planes']
    print(name, '발사당',per_launch,'kg, 전체',total,'kg')

5 planes 발사당 2400 kg, 전체 12000 kg
6 planes 발사당 2500 kg, 전체 15000 kg


## 확장 질문

- 같은 총 30기라도 한 궤도면 발사가 실패할 때 5면·6면의 성능 저하는 어떻게 다른가?
- 위상 `f`가 바뀌어도 RAAN은 같고 AoL만 바뀐다는 사실을 확인하라.
- 실제 분리장치와 adapter 질량을 포함하면 발사당 payload 여유는 얼마인가?